# Joint Fault Model — All Fault Parameters at Once

Combines the four separately-validated single-fault models into one function that takes
every fault probability as input simultaneously. **Read the caveat below before using this
for anything you'd defend under questioning** — the combination rule is a standard,
well-justified assumption, but it is not itself validated against real data where multiple
fault types fire at once, because (as far as this pipeline has collected so far) no such
joint-fault-type dataset exists yet for these four SDC fault types.

Run `20_data_cleaning_and_integrity.ipynb`, `21_toeplitz_finalkey_validation.ipynb`, and
`22_reconciliation_validation.ipynb` first (or just make sure `key_pairs_metadata.csv` and
the fault-model functions below are available).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from qne.cascade.finite_key import h, finite_key_output_length

RESULTS = PROJECT_DIR / "results"
key_pairs_df = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))


## The four validated per-fault-type models

Each of these was independently checked against real-channel data tonight/this week:
- **Toeplitz** and **final_key**: `p * (ell / (ell + t))`, chi-square NOT rejected in
  asymptotic and finite_key modes (p≈1.0).
- **reconciliation_state**: `1 - (1-p)^(n*QBER)`, replaces the rejected block-schedule model.
- **verify_digest**: structurally 0 -- this fault type never actually changes the key
  (confirmed: 0/N mismatches in the cleaned data). Include it in the interface for
  completeness/extensibility, but it contributes nothing to the combined probability.

In [ ]:
def asymptotic_key_length(n_bits, Q):
    return max(0, int(n_bits * (1 - 2 * h(Q))))

def verification_t(eps_ec=1e-10):
    return max(1, int(np.ceil(-np.log2(eps_ec))))

def ell_finite_key_actual(n, k_pe, Q):
    ell, r, t, nu = finite_key_output_length(n, k_pe, Q)
    return max(0, int(round(ell))), int(max(1, round(t)))

def ell_asymptotic_actual(n, k_pe, Q, eps=1e-10):
    return asymptotic_key_length(n, Q), verification_t(eps)


def p_toeplitz(krow, prob, ell_func=ell_asymptotic_actual):
    n_bits, k_pe, Q = int(krow["n_bits"]), int(krow["k_pe"]), float(krow["qber"])
    ell, t = ell_func(n_bits, k_pe, Q)
    return prob * (ell / (ell + t)) * 0.5

def p_final_key(krow, prob, ell_func=ell_asymptotic_actual):
    n_bits, k_pe, Q = int(krow["n_bits"]), int(krow["k_pe"]), float(krow["qber"])
    ell, t = ell_func(n_bits, k_pe, Q)
    return prob * (ell / (ell + t))

def p_reconciliation(krow, prob):
    n_bits, Q = int(krow["n_bits"]), float(krow["qber"])
    return 1 - (1 - prob) ** (n_bits * Q)

def p_verify_digest(krow, prob):
    return 0.0  # structural result: this fault type never changes the key

FAULT_MODELS = {
    "toeplitz_prob": p_toeplitz,
    "final_key_prob": p_final_key,
    "reconciliation_prob": p_reconciliation,
    "verify_digest_prob": p_verify_digest,
}


## The combination rule

For independently-firing faults, `P(no mismatch from any of them) = Π(1 - P_i)`, so
`P(at least one causes a mismatch) = 1 - Π(1 - P_i)`. Each fault type is injected by its own
independent random draw in a separate stage of the pipeline (PA extraction, reconciliation,
verification) with its own seed, so independence is a reasonable assumption -- but it is an
assumption, not something separately checked here.

In [ ]:
def predict_any_mismatch(krow, probs, ell_func=ell_asymptotic_actual):
    """probs: dict of {fault_name: firing_probability}. Any subset of
    FAULT_MODELS' keys -- omitted fault types are treated as prob=0 (off)."""
    p_no_mismatch = 1.0
    contributions = {}
    for name, prob in probs.items():
        if name not in FAULT_MODELS:
            raise ValueError(f"Unknown fault type '{name}' -- known types: {list(FAULT_MODELS)}")
        if name in ("toeplitz_prob", "final_key_prob"):
            p_i = FAULT_MODELS[name](krow, prob, ell_func)
        else:
            p_i = FAULT_MODELS[name](krow, prob)
        contributions[name] = p_i
        p_no_mismatch *= (1 - p_i)
    return 1 - p_no_mismatch, contributions


## Example: sweep a few realistic multi-fault scenarios

In [ ]:
krow = key_pairs_df.iloc[0]  # pick whichever key you want to illustrate with
print(f"Using key0: n_bits={int(krow['n_bits'])}, qber={float(krow['qber']):.4f}\n")

scenarios = [
    {"toeplitz_prob": 0.01, "final_key_prob": 0.0,  "reconciliation_prob": 0.0,  "verify_digest_prob": 0.0},
    {"toeplitz_prob": 0.0,  "final_key_prob": 0.0,  "reconciliation_prob": 0.05, "verify_digest_prob": 0.0},
    {"toeplitz_prob": 0.01, "final_key_prob": 0.01, "reconciliation_prob": 0.05, "verify_digest_prob": 0.1},
    {"toeplitz_prob": 0.05, "final_key_prob": 0.05, "reconciliation_prob": 0.1,  "verify_digest_prob": 0.2},
]

for probs in scenarios:
    p_any, contributions = predict_any_mismatch(krow, probs)
    print(f"probs={probs}")
    print(f"  per-fault-type contributions: {{k: f'{v:.4f}' for k, v in contributions.items()}}")
    print(f"  P(any mismatch) = {p_any:.4f}\n")


## What this is (and isn't) good for right now

- **Good for**: a single number to sweep in a "how does overall key-corruption risk scale
  as multiple fault mechanisms become more likely at once" plot, and for reasoning about
  which fault type dominates the risk in a given regime (the per-fault contributions are
  broken out above specifically so you can see that).
- **Not yet validated for**: claiming this is the *measured* joint mismatch rate under
  simultaneous multi-fault-type injection. That would need a real-channel collection where
  more than one of these four fault types fires in the same run, which doesn't appear to
  exist yet in your results directory (the joint sweep you already have -- polarization
  fidelity x packet loss -- is a different pair of fault types, on the classical/quantum
  channel axis, not these four SDC types).
- **To actually validate this combination rule**: collect even a modest real-channel batch
  with two fault types enabled simultaneously (e.g., toeplitz_prob=0.05 AND
  reconciliation_prob=0.05 together) and check whether the observed mismatch rate matches
  `predict_any_mismatch` before trusting it for anything you'd present as measured, rather
  than modeled.